Standard PRODUCTION GRADE Practice for Creating Short term Memory

In [47]:
# Generally in production, we keep in mind about the token we are sending to llm.
# when we maintani short term memory, it can become huge with time, and we know that directly or indireclty we are sending this whole memory to llm , causing huge money wastage and increase in latency.
# so what we can do is, we can use sliding window technique.
# we can keep last few messages, and we can generate a summary for the older message and replace this older message with its summary, in this way, we can keep our token low and at the same time we can have important knowledge as well.

In [48]:
from langgraph.graph import StateGraph, START, MessagesState, add_messages,END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.sqlite import SqliteStore 
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from typing import Annotated , Optional
from langchain_core.messages import HumanMessage, RemoveMessage
load_dotenv()

True

In [49]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

MAIN_LLM=ChatOpenAI (
    model= 'gpt-4o-mini',
    temperature=0.3,
    api_key=OPENAI_API_KEY,
    max_retries=3
)

FALLBACK_LLM=ChatGoogleGenerativeAI(
    model= 'gemini-1.5-turbo',
    temperature=0.3,
    api_key=GEMINI_API_KEY,
    max_retries=3
)


llm=MAIN_LLM.with_fallbacks([FALLBACK_LLM])

from langchain_core.callbacks import BaseCallbackHandler
class FallbackTracker(BaseCallbackHandler):
    def on_llm_error(self, error: BaseException, **kwargs) -> None:
        print(f"ChatGPT failed with error: {error}. Falling back to Gemini...")

# Pass the callback handler during invoke
tracker = FallbackTracker()

In [50]:
class GlobalState(BaseModel):
    messages:Annotated[list, add_messages]=[]
    summary: Optional[str] = Field(default=None,description="Summary of the messages.")
    
builder = StateGraph(GlobalState)

KEY CELLS  BELOW

In [51]:
def chatmodel(State: GlobalState):
    message=[]
    if State.summary:
        message.append(("ai",f"Conversation summary:\n {State.summary} "))


    message.extend(State.messages)
    print(f"current length of messages: {len(message)}")

    res=llm.invoke(message)
    return {"messages":  [("ai", res.content)]}

In [52]:
def summarize(State: GlobalState):
    existing_summary = State.summary

    if existing_summary:
        prompt = f'''Existing summary:\n{existing_summary}\n\n
            Extend the summary using the new conversation above.'''

    else:
        prompt = "Summarize the conversation above."

    message_with_summary=State.messages + [
        HumanMessage(content=prompt)
    ]

    res=llm.invoke(message_with_summary)

    #message to delete
    message_to_del=State.messages[:-2]
    return {
        "summary":res.content,
        "messages":[RemoveMessage(id=m.id) for m in message_to_del]
    
    }

In [53]:
def check_summarizer_eligibilty(State: GlobalState):
    if len(State.messages) > 6:
        return True

    else: return False

In [54]:
builder.add_node("chatmodel",chatmodel)
builder.add_node("summarize",summarize)

builder.add_edge(START,"chatmodel")
builder.add_conditional_edges(
    "chatmodel",
    check_summarizer_eligibilty,
    {
        True: "summarize",
        False: END
    }
)

In [60]:
import pprint
with SqliteSaver.from_conn_string("stm3.db") as memory:
    graph=builder.compile(checkpointer=memory)
    config={"configurable":{"thread_id":"stm3_01"}}
    user_input=input()
    events=graph.stream({"messages":[("user", user_input)]},config=config)
    for event in events:
        pprint.pprint(event)
    

current length of messages: 8
{'chatmodel': {'messages': [('ai', 'Your name is Saurav Sagar.')]}}
{'summarize': {'messages': [RemoveMessage(content='', additional_kwargs={}, response_metadata={}, id='2d06e4d0-e5c9-49b8-b90d-1dc7c43c5e77'),
                            RemoveMessage(content='', additional_kwargs={}, response_metadata={}, id='905c9517-a69f-44d0-8d1c-7539a6434366'),
                            RemoveMessage(content='', additional_kwargs={}, response_metadata={}, id='afb32745-249a-4a3b-990a-89ea2d86b769'),
                            RemoveMessage(content='', additional_kwargs={}, response_metadata={}, id='400d6662-5c2a-4b47-a48b-2e5dabcd79af'),
                            RemoveMessage(content='', additional_kwargs={}, response_metadata={}, id='a1aaba55-60b2-4f3f-b8dd-e4f8a9c05ff5'),
                            RemoveMessage(content='', additional_kwargs={}, response_metadata={}, id='7a2e6ee0-6ffd-40a1-a0fe-f83c78566ef6')],
               'summary': 'In the conversation, S